# 03. Model Training and Evaluation

Train classifiers with MLflow logging; pick best by validation F1.


In [ ]:
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline


In [ ]:
proc_dir = os.path.join("..", "data", "processed")
X_train = np.load(os.path.join(proc_dir, "X_train.npy"))
y_train = np.load(os.path.join(proc_dir, "y_train.npy"))
X_val = np.load(os.path.join(proc_dir, "X_val.npy"))
y_val = np.load(os.path.join(proc_dir, "y_val.npy"))
X_test = np.load(os.path.join(proc_dir, "X_test.npy"))
y_test = np.load(os.path.join(proc_dir, "y_test.npy"))

with open(os.path.join(proc_dir, "label_classes.json"), encoding="utf-8") as f:
    target_names = json.load(f)
print("Shapes — train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)


In [ ]:
MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("iris-classification")
print("MLflow tracking URI:", mlflow.get_tracking_uri())


In [ ]:
def evaluate_model(model, X, y):
    """Return accuracy and macro F1."""
    pred = model.predict(X)
    return {
        "accuracy": accuracy_score(y, pred),
        "f1_macro": f1_score(y, pred, average="macro"),
    }


def plot_confusion_matrix(model, X, y, labels, title="Confusion matrix"):
    pred = model.predict(X)
    cm = confusion_matrix(y, pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    return fig


In [ ]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
}

results_rows = []
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        val_metrics = evaluate_model(model, X_val, y_val)
        train_metrics = evaluate_model(model, X_train, y_train)
        mlflow.log_param("model", name)
        mlflow.log_metrics({f"val_{k}": v for k, v in val_metrics.items()})
        mlflow.log_metrics({f"train_{k}": v for k, v in train_metrics.items()})
        mlflow.sklearn.log_model(model, artifact_path="model")
        results_rows.append(
            {
                "model": name,
                "val_accuracy": val_metrics["accuracy"],
                "val_f1": val_metrics["f1_macro"],
                "train_accuracy": train_metrics["accuracy"],
                "train_f1": train_metrics["f1_macro"],
            }
        )
        print(name, val_metrics)

results_df = pd.DataFrame(results_rows).sort_values("val_f1", ascending=False)
results_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
results_df.plot(x="model", y=["val_accuracy", "val_f1"], kind="bar", ax=ax, rot=0)
ax.set_title("Validation metrics by model")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]
print("Best model (by val_f1):", best_name)

# Refit best model on train+val for final artifact (optional); here evaluate on test as trained on train only
test_metrics = evaluate_model(best_model, X_test, y_test)
y_pred = best_model.predict(X_test)
print("\nTest metrics:", test_metrics)
print("\nClassification report (test):\n")
print(classification_report(y_test, y_pred, target_names=target_names))


In [ ]:
fig = plot_confusion_matrix(best_model, X_test, y_test, target_names, title=f"Test — confusion matrix ({best_name})")
plt.show()

models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)
best_path = os.path.join(models_dir, "best_model.pkl")
with open(best_path, "wb") as f:
    pickle.dump(best_model, f)
meta = {
    "best_model_name": best_name,
    "model_path": best_path,
    "val_f1": float(results_df.iloc[0]["val_f1"]),
    "test_f1_macro": float(test_metrics["f1_macro"]),
}
with open(os.path.join(models_dir, "best_model_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print("Saved:", best_path)
